<a href="https://colab.research.google.com/github/prateekraj91/gpt-learning/blob/main/gpt_learning_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wandb

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import wandb

# -----------------------------
# wandb setup
# -----------------------------

wandb.login()

wandb.init(
    project="mini-gpt-training"
)

# -----------------------------
# Attention Head
# -----------------------------

class Head(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()

        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)

        self.register_buffer(
            'mask',
            torch.tril(torch.ones(256, 256))
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        scores = q @ k.transpose(-2, -1) / (C ** 0.5)

        scores = scores.masked_fill(
            self.mask[:T, :T] == 0,
            float('-inf')
        )

        weights = F.softmax(scores, dim=-1)

        out = weights @ v

        return out

# -----------------------------
# Multi Head Attention
# -----------------------------

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.head_size = d_model // num_heads

        self.heads = nn.ModuleList([
            Head(d_model, self.head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):

        out = torch.cat([h(x) for h in self.heads], dim=-1)

        out = self.proj(out)

        return out

# -----------------------------
# Feed Forward
# -----------------------------

class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):

        return self.net(x)

# -----------------------------
# Transformer Block
# -----------------------------

class Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, num_heads)

        self.ffwd = FeedForward(d_model)

        self.ln1 = nn.LayerNorm(d_model)

        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):

        x = x + self.attn(self.ln1(x))

        x = x + self.ffwd(self.ln2(x))

        return x

# -----------------------------
# GPT Model
# -----------------------------

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.pos_embedding = nn.Embedding(256, d_model)

        self.blocks = nn.Sequential(
            *[
                Block(d_model, num_heads)
                for _ in range(num_layers)
            ]
        )

        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        B, T = x.shape

        token_emb = self.token_embedding(x)

        pos_emb = self.pos_embedding(
            torch.arange(T, device=x.device)
        )

        x = token_emb + pos_emb

        x = self.blocks(x)

        logits = self.lm_head(x)

        return logits

# -----------------------------
# Model Setup
# -----------------------------

vocab_size = 1000
batch_size = 2
seq_len = 8

model = GPT(
    vocab_size=vocab_size,
    d_model=32,
    num_heads=4,
    num_layers=3
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

# -----------------------------
# Training Loop
# -----------------------------

for epoch in range(20):

    # fake dataset
    x = torch.randint(
        0,
        vocab_size,
        (batch_size, seq_len)
    )

    targets = torch.randint(
        0,
        vocab_size,
        (batch_size, seq_len)
    )

    # forward pass
    logits = model(x)

    B, T, C = logits.shape

    logits = logits.view(B * T, C)

    targets = targets.view(B * T)

    # loss
    loss = F.cross_entropy(logits, targets)

    # perplexity
    ppl = torch.exp(loss)

    # backward
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    scheduler.step()

    # fake validation loss
    val_loss = loss + torch.rand(1).item() * 0.1

    # wandb logging
    wandb.log({
        "train_loss": loss.item(),
        "val_loss": val_loss.item(),
        "perplexity": ppl.item(),
        "learning_rate": scheduler.get_last_lr()[0]
    })

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {loss.item():.4f} | "
        f"Val Loss: {val_loss.item():.4f} | "
        f"PPL: {ppl.item():.4f} | "
        f"LR: {scheduler.get_last_lr()[0]:.6f}"
    )

wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: prateekraj91 (prateekraj91-bits-pil) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1 | Loss: 7.2566 | Val Loss: 7.3324 | PPL: 1417.4586 | LR: 0.000298
Epoch 2 | Loss: 7.4925 | Val Loss: 7.5460 | PPL: 1794.4966 | LR: 0.000293
Epoch 3 | Loss: 7.4527 | Val Loss: 7.4967 | PPL: 1724.4771 | LR: 0.000284
Epoch 4 | Loss: 7.0929 | Val Loss: 7.1784 | PPL: 1203.4015 | LR: 0.000271
Epoch 5 | Loss: 7.5240 | Val Loss: 7.5828 | PPL: 1851.9757 | LR: 0.000256
Epoch 6 | Loss: 7.5732 | Val Loss: 7.6253 | PPL: 1945.4264 | LR: 0.000238
Epoch 7 | Loss: 7.2587 | Val Loss: 7.2747 | PPL: 1420.4493 | LR: 0.000218
Epoch 8 | Loss: 7.2371 | Val Loss: 7.3351 | PPL: 1390.0876 | LR: 0.000196
Epoch 9 | Loss: 7.2471 | Val Loss: 7.3407 | PPL: 1404.0271 | LR: 0.000173
Epoch 10 | Loss: 7.0374 | Val Loss: 7.0984 | PPL: 1138.3997 | LR: 0.000150
Epoch 11 | Loss: 7.5726 | Val Loss: 7.6463 | PPL: 1944.1736 | LR: 0.000127
Epoch 12 | Loss: 7.3756 | Val Loss: 7.4689 | PPL: 1596.5294 | LR: 0.000104
Epoch 13 | Loss: 7.1997 | Val Loss: 7.2612 | PPL: 1339.0671 | LR: 0.000082
Epoch 14 | Loss: 7.0706 | Val Loss

learning_rate,███▇▇▇▆▆▅▅▄▃▃▂▂▂▁▁▁▁
perplexity,▄▇▆▃▇█▄▄▄▃█▆▄▃▂▅█▄▆▁
train_loss,▅▇▇▄▇█▅▅▅▃█▆▅▃▃▅█▄▆▁
val_loss,▅▇▇▄▇█▅▅▅▃█▆▄▃▂▅█▅▆▁
learning_rate,0
perplexity,894.75702
train_loss,6.79655
val_loss,6.89292
